# AML Alert Prioritization — Solution & Reproducible Pipeline
**Team:** Мясокомбинат  
**Team ID:** 6927C48E  
**Task:** Binary classification of AML alerts (`eskalatsiya = 1` vs `0`)  
**Evaluation Metric:** ROC-AUC  
**Model:** 5-Fold Stratified LightGBM Gradient Boosting with Relational Feature Aggregation


## 1. Imports and Environment Setup


In [1]:
import os
import time
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

print("Environment initialized.")


Environment initialized.


## 2. Data Loading & Exploratory Verification
Load both signals and transaction streams for training and testing.
The relational transaction table is aggregated at the `signal_id` level.


In [2]:
train_signals = pd.read_csv('train_signals.csv')
train_trans = pd.read_parquet('train_transactions.parquet')
test_signals = pd.read_csv('test_signals.csv')
test_trans = pd.read_parquet('test_transactions.parquet')

print(f"Train signals: {len(train_signals):,}")
print(f"Train transactions: {len(train_trans):,}")
print(f"Test signals: {len(test_signals):,}")
print(f"Test transactions: {len(test_trans):,}")

esc_rate = train_signals['eskalatsiya'].mean()
print(f"Train Escalation Rate: {esc_rate * 100:.2f}% (severe class imbalance)")


Train signals: 14,000
Train transactions: 6,987,663
Test signals: 6,000
Test transactions: 3,027,575
Train Escalation Rate: 17.18% (severe class imbalance)


## 3. Feature Engineering Pipeline
Transform relational transactional streams into behavioral portraits per `signal_id`:
- **Volume & count aggregates:** count, sum, mean, std, min, max of `miqdor_indeksi`.
- **Directional pivots:** frequency and sum of `kirim` (inbound) and `chiqim` (outbound).
- **Ratios:** `chiqim_kirim_sum_ratio`, `chiqim_kirim_cnt_ratio` to capture layering & pass-through transit accounts.
- **Channel pivots:** distribution across `karta`, `bank_otkazmasi`, `naqd`, `xalqaro`.
- **Temporal recency & duration:** `days_to_last_txn`, `days_to_first_txn`, `txn_span_days`.
- **Calendar features:** `signal_month`, `signal_day`, `signal_dow`.


In [3]:
def engineer_features(signals_df, trans_df):
    signals = signals_df.copy()
    trans = trans_df.copy()

    signals['signal_sanasi'] = pd.to_datetime(signals['signal_sanasi'])
    trans['tranzaksiya_vaqti'] = pd.to_datetime(trans['tranzaksiya_vaqti'])
    trans['kirim_chiqim'] = trans['kirim_chiqim'].astype(str)
    trans['tranzaksiya_turi'] = trans['tranzaksiya_turi'].astype(str)

    # 1. Base transaction statistics per signal
    base_stats = trans.groupby('signal_id').agg(
        txn_count=('miqdor_indeksi', 'count'),
        amount_sum=('miqdor_indeksi', 'sum'),
        amount_mean=('miqdor_indeksi', 'mean'),
        amount_std=('miqdor_indeksi', 'std'),
        amount_max=('miqdor_indeksi', 'max'),
        amount_min=('miqdor_indeksi', 'min'),
        last_txn_time=('tranzaksiya_vaqti', 'max'),
        first_txn_time=('tranzaksiya_vaqti', 'min')
    ).reset_index()

    # 2. Directional flow pivot (inflow vs outflow)
    dir_piv = trans.pivot_table(
        index='signal_id',
        columns='kirim_chiqim',
        values='miqdor_indeksi',
        aggfunc=['count', 'sum'],
        fill_value=0
    )
    dir_piv.columns = [f"{col[1]}_{col[0]}" for col in dir_piv.columns]
    dir_piv = dir_piv.reset_index()

    # 3. Channel breakdown pivot
    type_piv = trans.pivot_table(
        index='signal_id',
        columns='tranzaksiya_turi',
        values='miqdor_indeksi',
        aggfunc=['count', 'sum'],
        fill_value=0
    )
    type_piv.columns = [f"{col[1]}_{col[0]}" for col in type_piv.columns]
    type_piv = type_piv.reset_index()

    # 4. Relational merge
    merged = base_stats.merge(dir_piv, on='signal_id', how='left').merge(type_piv, on='signal_id', how='left')
    output = signals.merge(merged, on='signal_id', how='left')

    # 5. Temporal recency and window duration
    output['days_to_last_txn'] = (output['signal_sanasi'] - output['last_txn_time']).dt.total_seconds() / 86400.0
    output['days_to_first_txn'] = (output['signal_sanasi'] - output['first_txn_time']).dt.total_seconds() / 86400.0
    output['txn_span_days'] = (output['last_txn_time'] - output['first_txn_time']).dt.total_seconds() / 86400.0

    # 6. Flow velocity ratios
    if 'chiqim_sum' in output.columns and 'kirim_sum' in output.columns:
        output['chiqim_kirim_sum_ratio'] = output['chiqim_sum'] / (output['kirim_sum'] + 1e-5)
    if 'chiqim_count' in output.columns and 'kirim_count' in output.columns:
        output['chiqim_kirim_cnt_ratio'] = output['chiqim_count'] / (output['kirim_count'] + 1e-5)

    # 7. Calendar attributes
    output['signal_month'] = output['signal_sanasi'].dt.month
    output['signal_day'] = output['signal_sanasi'].dt.day
    output['signal_dow'] = output['signal_sanasi'].dt.dayofweek

    cols_to_drop = ['last_txn_time', 'first_txn_time']
    output = output.drop(columns=[c for c in cols_to_drop if c in output.columns])

    return output

print("Feature engineering function defined.")


Feature engineering function defined.


## 4. Feature Extraction on Train and Test Sets


In [4]:
t0 = time.time()
print("Engineering training features...")
train_df = engineer_features(train_signals, train_trans)

print("Engineering test features...")
test_df = engineer_features(test_signals, test_trans)

exclude_cols = ['signal_id', 'signal_sanasi', 'eskalatsiya']
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

for col in feature_cols:
    if col not in test_df.columns:
        test_df[col] = 0

X = train_df[feature_cols].fillna(0)
y = train_df['eskalatsiya']
X_test = test_df[feature_cols].fillna(0)

print(f"Features extracted ({len(feature_cols)} total) in {time.time() - t0:.1f}s.")
print("Feature columns:", feature_cols)


Engineering training features...


Engineering test features...


Features extracted (26 total) in 3.5s.
Feature columns: ['txn_count', 'amount_sum', 'amount_mean', 'amount_std', 'amount_max', 'amount_min', 'chiqim_count', 'kirim_count', 'chiqim_sum', 'kirim_sum', 'bank_otkazmasi_count', 'karta_count', 'naqd_count', 'xalqaro_count', 'bank_otkazmasi_sum', 'karta_sum', 'naqd_sum', 'xalqaro_sum', 'days_to_last_txn', 'days_to_first_txn', 'txn_span_days', 'chiqim_kirim_sum_ratio', 'chiqim_kirim_cnt_ratio', 'signal_month', 'signal_day', 'signal_dow']


## 5. Model Architecture & 5-Fold Stratified Cross-Validation
- **Algorithm:** LightGBM Gradient Boosting (`gbdt`)
- **Validation:** 5-Fold Stratified K-Fold (preserves ~16.7% escalation proportion per fold)
- **Early Stopping:** 50 rounds to prevent overfitting
- **Metric:** ROC-AUC


In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'n_estimators': 1500,
    'random_state': 42,
    'verbose': -1
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f"Fold {fold+1} ROC-AUC: {fold_auc:.5f}")

    test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

cv_auc = roc_auc_score(y, oof_preds)
print("-----------------------------------------")
print(f"Overall 5-Fold Out-Of-Fold CV ROC-AUC: {cv_auc:.5f}")
print("-----------------------------------------")


Fold 1 ROC-AUC: 0.62408
Fold 2 ROC-AUC: 0.61224


Fold 3 ROC-AUC: 0.59848


Fold 4 ROC-AUC: 0.60647


Fold 5 ROC-AUC: 0.59215
-----------------------------------------
Overall 5-Fold Out-Of-Fold CV ROC-AUC: 0.60312
-----------------------------------------


## 6. Official Prediction Export (`team_6927C48E.csv`)
Generating predictions for `test_signals.csv` strictly adhering to competition formatting rules.


In [6]:
submission = pd.DataFrame({
    'signal_id': test_df['signal_id'],
    'ehtimollik': np.clip(test_preds, 0.0, 1.0)
})

submission.to_csv('team_6927C48E.csv', index=False)
print(f"Submission successfully written to team_6927C48E.csv with {len(submission):,} rows.")


Submission successfully written to team_6927C48E.csv with 6,000 rows.


## 7. Submission Format Verification
Verification against all hackathon submission criteria:
1. Exactly two columns: `signal_id,ehtimollik` in order
2. Exactly one row per test signal (6,000 rows)
3. No duplicate IDs, missing IDs, or NaNs
4. Valid probability bounds: 0 <= ehtimollik <= 1
5. No index column


In [7]:
# Automated verification check
sub = pd.read_csv('team_6927C48E.csv')
test = pd.read_csv('test_signals.csv')

assert list(sub.columns) == ['signal_id', 'ehtimollik'], 'Column mismatch!'
assert len(sub) == len(test), f'Expected {len(test)} rows, got {len(sub)}'
assert sub['signal_id'].duplicated().sum() == 0, 'Duplicate IDs found!'
assert sub['signal_id'].isna().sum() == 0, 'NaNs in signal_id!'
assert sub['ehtimollik'].isna().sum() == 0, 'NaNs in ehtimollik!'
assert sub['ehtimollik'].min() >= 0.0 and sub['ehtimollik'].max() <= 1.0, 'Out-of-bounds probabilities!'
assert (sub['signal_id'].values == test['signal_id'].values).all(), 'ID ordering mismatch!'

print('SUCCESS: All 8 submission criteria verified and passed!')
print(sub.head(10))


SUCCESS: All 8 submission criteria verified and passed!
   signal_id  ehtimollik
0  SG_000001    0.156720
1  SG_000007    0.173055
2  SG_000009    0.136595
3  SG_000010    0.130092
4  SG_000011    0.123159
5  SG_000018    0.213670
6  SG_000024    0.184022
7  SG_000026    0.152292
8  SG_000032    0.190880
9  SG_000033    0.241439
